# NB_hospital_daily_summary

Builds the **gold.hospital_daily_summary** table — one row per hospital per calendar day with test volumes, abnormal/critical rates, and day-over-day trend via `LAG()`. Full rebuild every run.

In [0]:
# -- Configuration ----------------------------------------------------------
catalog = "adb_databeli"
target_table = f"{catalog}.gold.hospital_daily_summary"

print(f"✅ Configuration")
print(f"   📌 Catalog      : {catalog}")
print(f"   📌 Target table : {target_table}")
print(f"   📌 Load type    : FULL (CREATE OR REPLACE)")

In [0]:
%sql
CREATE OR REPLACE TABLE adb_databeli.gold.hospital_daily_summary AS

WITH base AS (
    SELECT
        lr.hospital_id,
        h.hospital_name,
        lr.test_date                                                    AS summary_date,
        c.day_name,
        c.is_weekend,
        COUNT(*)                                                        AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                    AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                     AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
    FROM adb_databeli.silver.lab_results  lr
    LEFT JOIN adb_databeli.silver.hospitals  h
        ON lr.hospital_id = h.hospital_id
    LEFT JOIN adb_databeli.silver.calendar  c
        ON c.date_key = INT(DATE_FORMAT(lr.test_date, 'yyyyMMdd'))
    GROUP BY lr.hospital_id, h.hospital_name, lr.test_date, c.day_name, c.is_weekend
)

SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2))   AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2))   AS critical_rate_pct,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)     AS prev_day_total_tests,
    CAST(
        ROUND(
            CASE
                WHEN LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) IS NULL
                  OR LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) = 0
                THEN NULL
                ELSE (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
                     / CAST(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS DOUBLE) * 100
            END, 2
        ) AS DECIMAL(6,2)
    )                                                                          AS test_volume_change_pct
FROM base

In [0]:
row_count = spark.table(target_table).count()
print(f"✅ Table '{target_table}' built successfully")
print(f"   📊 Total rows: {row_count:,}")

display(spark.table(target_table).limit(10))

In [0]:
dbutils.notebook.exit(str(row_count))